#### ***Creating Input-Target Pairs***
##### **We want the model to learn from the Input and predict the next token.**

In [51]:
## tiktoken is a python library is used to convert the text into tokens,
#  and also convert those tokens into again text.
import tiktoken

In [52]:
## let's read the data from a file
with open("../Data/the-verdict.txt","r",encoding = "utf-8") as f:
    raw_text = f.read()

In [53]:
### I am using gpt2
tokenizer = tiktoken.get_encoding("gpt2")

enc_txt = tokenizer.encode(raw_text)
print(len(enc_txt))

5145


In [54]:
## based on the results total number of tokens are 5145 the training set, 
# after applying BPE Tokenizer


In [55]:
### Let's remove the first 50 tokens to make more chruchy

enc_sample = enc_txt[50:]
print(len(enc_sample))

5095


In [56]:
### Let's split data into input and target variables by using context size.

## How it's work
# step:1 suppose the let's x = [1,2,3,4] as input
# step:2 y = [2,3,4,5] as output, why because of every time we need predict one word at a time
# so that 1->2, 1,2->3, 1,2,3->4, 1,2,3,4->5


context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"X: {x}")
print(f"Y:      {y}")


X: [290, 4920, 2241, 287]
Y:      [4920, 2241, 287, 257]


In [57]:
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context,"--->",desired)

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


In [58]:
### Left side of Arrow Represents the Input, and right side as output

In [59]:
### Lets decode and represents as word
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context),"--->",tokenizer.decode([desired]))

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


#### ***Implementing a Data Loader***

In [60]:
## We will use Pytorch built-in for dataset and dataloader classes to implement a Data Loader.

#How we are implementing
#1.Tokenize the Entire Text.
#2.Using sliding window means based on context size split's the data into input and output variable,
#output = input+[1] is used to predict the next token by one
#Return the Entire Dataset.
#Return the Single Row from the Dataset.

In [70]:
from torch.utils.data import DataLoader,Dataset
import torch


class GPTDatasetV1(Dataset):
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text,allowed_special={"<|endoftext|>"})

        #splits the token ids into input and output token ids based on context size
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    ## to get particular row as input and output
    def  __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]

In [71]:
### Batch Size Tells about the Number Of CPU's 
### num_workers:number of threads for each cpu
## drop_last : prevent the data if less than max_length

In [72]:
def create_dataloader_v1(txt, batch_size = 4,max_length = 256,
                        stride = 128,shuffle = True,drop_last=True,
                         num_workers = 0):

    #Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    #create a dataset.
    dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)

    ## Create a Data Loader
    dataloader = DataLoader(dataset,batch_size=batch_size,
                            shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)

    return dataloader
                        

In [73]:
## dataloader

dataloader = create_dataloader_v1(raw_text,batch_size=1,max_length=4,stride=1,shuffle=False)

### first batch input and output from the dataset
batch_iter = iter(dataloader)

first_batch = next(batch_iter)
print("First Batch:",first_batch)

First Batch: [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [74]:
## second batch
second_batch = next(batch_iter)
print("Second Batch:",second_batch)

Second Batch: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [75]:
## third batch
third_batch = next(batch_iter)
print("Third Batch:",third_batch)

Third Batch: [tensor([[2885, 1464, 1807, 3619]]), tensor([[1464, 1807, 3619,  402]])]


In [76]:
## dataloader

dataloader = create_dataloader_v1(raw_text,batch_size=4,max_length=4,stride=4,shuffle=False)

### first batch input and output from the dataset
batch_iter = iter(dataloader)

inputs,targets = next(batch_iter)
print(inputs)
print(targets)

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257]])
tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922]])
